# Objective 3 — Notebook 05: Baseline Model Development

**Project:** Evaluating Multi-Source Data Fusion and Uncertainty in Hyper-Local Flash-Flood Prediction for Hilly Regions of India (Uttarakhand)  
**Objective 3:** Initial benchmark of candidate models: Logistic Regression, Random Forest, and XGBoost.

---

### Core Requirements:
1. **Chronological Splitting**: Partition into Train, Validation, and untouched Test.
2. **Rare Event Analysis**: Measure class imbalance and positive event frequency.
3. **Baseline Pipeline Fitting**: Evaluate baseline models on default parameters with train-only preprocessing.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
from src.data_loader import load_ml_dataset, FEATURE_EXPERIMENTS
from src.split import get_chronological_splits, compute_rare_event_stats
from src.preprocessing import get_logistic_preprocessor, get_tree_preprocessor
from src.evaluate import compute_all_metrics
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

print("Baseline modeling dependencies loaded.")

## 1. Load Dataset and Chronological Partition

In [ ]:
df = load_ml_dataset()
train_df, val_df, test_df = get_chronological_splits(df)

print(f"Total: {len(df):,} | Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
display(pd.DataFrame([
    compute_rare_event_stats(train_df, "Train"),
    compute_rare_event_stats(val_df, "Validation"),
    compute_rare_event_stats(test_df, "Test"),
]))

## 2. Train Default Baseline Models

In [ ]:
features = FEATURE_EXPERIMENTS["Exp_E_Full_Multimodal"]
X_train, y_train = train_df[features], train_df["flood_event_24h"]
X_val, y_val = val_df[features], val_df["flood_event_24h"]

# 1. Logistic Regression
lr = Pipeline([("prep", get_logistic_preprocessor()), ("clf", LogisticRegression(class_weight="balanced", random_state=42, max_iter=1000))])
lr.fit(X_train, y_train)
lr_val_p = lr.predict_proba(X_val)[:, 1]

# 2. Random Forest
rf = Pipeline([("prep", get_tree_preprocessor()), ("clf", RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1))])
rf.fit(X_train, y_train)
rf_val_p = rf.predict_proba(X_val)[:, 1]

# 3. XGBoost
xgb = Pipeline([("prep", get_tree_preprocessor()), ("clf", XGBClassifier(eval_metric="logloss", random_state=42, n_jobs=-1))])
xgb.fit(X_train, y_train)
xgb_val_p = xgb.predict_proba(X_val)[:, 1]

results = [
    {"model": "Logistic Regression (Baseline)", **compute_all_metrics(y_val.values, lr_val_p)},
    {"model": "Random Forest (Baseline)", **compute_all_metrics(y_val.values, rf_val_p)},
    {"model": "XGBoost (Baseline)", **compute_all_metrics(y_val.values, xgb_val_p)},
]
display(pd.DataFrame(results))